# vLLM vs Hugging Face transformers on a free Colab T4

This notebook runs the whole benchmark on a Colab T4 GPU. The code lives in
[the GitHub repo](https://github.com/isarinio22/vllm-free-gpu-benchmark), and this notebook clones it.

**Connect to a T4 first.** In VS Code with the Google Colab extension: *Select Kernel → Colab →
New Colab Server → GPU → T4*. In the browser: *Runtime → Change runtime type → T4 GPU*.

Then run all cells. It takes roughly 30–40 minutes. Each benchmark runs as its own `!python`
process: vLLM reserves most of the GPU memory up front, so HF and vLLM never share a process.

In [ ]:
!nvidia-smi

## 1. Get the code

In [ ]:
import os

REPO_URL = "https://github.com/isarinio22/vllm-free-gpu-benchmark"
REPO_DIR = "/content/vllm-free-gpu-benchmark"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

## 2. Install pinned requirements (about 3–5 minutes)

pip may warn about conflicts with other packages preinstalled on Colab. This benchmark doesn't
use those packages. You don't need to restart the runtime: every benchmark runs in a fresh process
that picks up the new versions.

In [ ]:
!pip install -q -r requirements.txt

## 3. Build the fixed prompt set (200 Dolly-15k prompts, seed 42)

In [ ]:
!python src/prompts.py

## 4. Experiment 1: offline throughput

Each method does 1 discarded warm-up run and 5 timed runs. `hf_naive` is the slow one (about 12 minutes).

In [ ]:
!python src/offline_bench.py --method hf_naive

In [ ]:
!python src/offline_bench.py --method hf_batched

In [ ]:
# If vLLM fails with an attention-kernel error on the T4, add: --attention-backend TRITON_ATTN
!python src/offline_bench.py --method vllm

## 5. Experiment 2: online serving load test

Starts `vllm serve` in the background, then sends streaming requests at concurrency 1, 4, 16 and 64.
The server log is written to `results/raw/vllm_server.log`.

In [ ]:
!python src/load_test.py

## 6. Results

Copy the two CSVs printed below into your local `results/` folder, then run `python src/plot.py`
locally to regenerate the charts.

In [ ]:
!python src/summarize.py && python src/plot.py

from IPython.display import Image, display

for name in ("offline_throughput", "load_test"):
    path = f"results/{name}.png"
    if os.path.exists(path):
        display(Image(path))